# BattSafe AI Model Training

This notebook trains an Isolation Forest on normal sensor readings, evaluates anomalies, and exports an embedded baseline configuration. The bundled dataset is synthetic and is only for checking the workflow. Replace it with calibrated real data before drawing conclusions.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib


## 1. Upload or load the dataset

Expected columns: `temperature_c`, `battery_voltage_v`, `gas_adc`, `condition`. The `condition` column should contain `normal` or `anomaly`. Training uses only normal rows.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import json, joblib
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

uploaded = files.upload()  # Upload battsafe_sensor_data.csv or your own CSV
csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if csv_name is None:
    raise ValueError("Please upload a CSV file.")

df = pd.read_csv(csv_name)
FEATURES = ["temperature_c", "battery_voltage_v", "gas_adc"]
required = FEATURES + ["condition"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

df = df.dropna(subset=required).copy()
df["condition"] = df["condition"].astype(str).str.lower()
print("Rows:", len(df))
display(df.head())
print(df["condition"].value_counts())


## 2. Train and evaluate the Isolation Forest

Use this result as an educational prototype evaluation only. If the CSV is synthetic, the metrics do not represent real-world EV safety performance.

In [ ]:
normal = df[df["condition"] == "normal"]
if len(normal) < 20:
    raise ValueError("At least 20 normal rows are recommended.")

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42
)
model.fit(normal[FEATURES].astype(float))

X_all = df[FEATURES].astype(float)
y_true = (df["condition"] != "normal").astype(int)
y_pred = (model.predict(X_all) == -1).astype(int)

print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=["normal", "anomaly"],
    zero_division=0
))
print("Confusion matrix [normal, anomaly]:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

joblib.dump(model, "Battsafe_isolation_forest.joblib")
print("Saved Python model.")


## 3. Export a lightweight configuration for ESP32

The ESP32 firmware below does not run the Python Isolation Forest. Instead, it uses baseline means and standard deviations exported from normal readings to calculate a lightweight z-score anomaly indicator. This is a separate embedded approximation.

In [ ]:
means = normal[FEATURES].astype(float).mean()
stds = normal[FEATURES].astype(float).std(ddof=0).replace(0, 1e-6)

config = {
    "model_name": "Battsafe embedded z-score baseline",
    "features": FEATURES,
    "means": {k: float(means[k]) for k in FEATURES},
    "stds": {k: float(stds[k]) for k in FEATURES},
    "z_limit": 3.0,
    "warning_note": "Demonstration only; calibrate with real sensor data."
}
with open("Battsafe_model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(json.dumps(config, indent=2))


## 4. Download generated model files

In [ ]:
files.download("Battsafe_model_config.json")
files.download("Battsafe_isolation_forest.joblib")
